<a href="https://colab.research.google.com/github/DELEnomore/LLM/blob/colab/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [2]:
import os.path
from abc import abstractmethod
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer, pipeline
import os
from google.colab import drive, userdata
import logging
from huggingface_hub import login

In [3]:
os.environ["WANDB_DISABLED"] = "true"

In [4]:
GOOGLE_DRIVE_WORKSPACE_DIR = 'drive/MyDrive/colab_workspace/LLM'
_CACHE_DIR = GOOGLE_DRIVE_WORKSPACE_DIR + '/cache'
MODEL_CACHE_DIR = _CACHE_DIR + '/model'
DATASET_CACHE_DIR = _CACHE_DIR + '/dataset'
# MODEL_NAME = "meta-llama/Llama-3.2-1B"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MODEL_CHECKPOINT_DIR = GOOGLE_DRIVE_WORKSPACE_DIR + '/model_output/' + MODEL_NAME
MODEL_OUTPUT_DIR = MODEL_CHECKPOINT_DIR + '/best_model'



In [ ]:
# 连接Google Drive。不使用可忽略这一行
drive.mount('/content/drive')

In [6]:
# secret saved by Google Colab
login(token=userdata.get('hugging_face_token'))

In [7]:
def format_chatml(input, output):
    messages = []
    if input:
        messages.append({"role": "user", "content": input})
    if output:
        messages.append({"role": "assistant", "content": output})
    return messages

In [8]:
def batch_format_chatml(batch_input, batch_output):
    batch_message = []
    for input, output in zip(batch_input, batch_output):
        batch_message.append(format_chatml(input, output))

    return batch_message

In [9]:
class DatasetInterface:
    PATH = ''

    NAME = None

    def __init__(self, tokenizer):
        self.data = load_dataset(self.PATH, self.NAME, cache_dir=DATASET_CACHE_DIR)
        self.tokenizer = tokenizer

    def get_data(self):
        return self.data

    @abstractmethod
    def _batch_get_input_and_output(self, example):
        pass

    def tokenize_function_4_casual_lm(self, example):
        input, output = self._batch_get_input_and_output(example)
        chatml = batch_format_chatml(input, output)
        formated_input = tokenizer.apply_chat_template(
            chatml,
            tokenize=False,
        )
        model_input = tokenizer(formated_input, padding="max_length", truncation=True, max_length=128)
        model_input["labels"] = model_input["input_ids"].copy()
        return model_input

In [10]:
class TestDataset(DatasetInterface):

    PATH = 'BeingIsA/test'

    def _batch_get_input_and_output(self, examples):
        inputs = []
        labels = []
        for conversation in examples['conversations']:
            dialog_input = ""
            dialog_output = ""
            for turn in conversation:
                if turn["from"] == "human":
                    dialog_input = turn["value"]
                elif turn["from"] == "gpt":
                    dialog_output = turn["value"]
            inputs.append(dialog_input.strip())
            labels.append(dialog_output.strip())

        return inputs, labels


In [11]:
class HuatuoDataset(DatasetInterface):

    PATH = "FreedomIntelligence/HuatuoGPT2-Pretraining-Instruction"

    NAME = "Meidcal_Encyclopedia_cn"

    def _batch_get_input_and_output(self, examples):
        inputs = []
        labels = []
        for conversation in examples['conversations']:
            dialog_input = ""
            dialog_output = ""
            for turn in conversation:
                if turn["from"] == "human":
                    dialog_input = turn["value"]
                elif turn["from"] == "gpt":
                    dialog_output = turn["value"]
            inputs.append(dialog_input.strip())
            labels.append(dialog_output.strip())

        return inputs, labels

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME, cache_dir=MODEL_CACHE_DIR)
tokenizer.pad_token = tokenizer.eos_token

# 修改数据集在这里
dataset_instance = HuatuoDataset(tokenizer)
data = dataset_instance.get_data()

tokenized_dataset = data.map(dataset_instance.tokenize_function_4_casual_lm, num_proc=4, batched=True)
splited_dataset = tokenized_dataset['train'].train_test_split(test_size=0.2)
train_dataset = splited_dataset['train']
val_dataset = splited_dataset['test']
print(f'train dataset size: {len(train_dataset)}')
print(f'val dataset size: {len(val_dataset)}')

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,  # 混合精度
    device_map="auto",  # 自动分配到 GPU
    cache_dir=MODEL_CACHE_DIR
).to("cuda")


In [ ]:
lora_config = LoraConfig(
    r=8,  # LoRA 的秩
    lora_alpha=32,  # LoRA 的缩放因子
    lora_dropout=0.05,  # Dropout 概率
    bias="none",  # LoRA bias 设置
    task_type="CAUSAL_LM",  # 任务类型：自回归文本生成
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    # 如果需要根据具体模型结构，调整 target_modules
)

peft_model = get_peft_model(model, lora_config).to("cuda")
peft_model.print_trainable_parameters()  # 查看可训练参数量

def check_point_exists():
    if [os.path.join(MODEL_CHECKPOINT_DIR, d) for d in os.listdir(MODEL_CHECKPOINT_DIR) if d.startswith("checkpoint")]:
        return True
    return False

training_args = TrainingArguments(
    output_dir=MODEL_CHECKPOINT_DIR,
    overwrite_output_dir=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=500,
    evaluation_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    report_to="none",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,

)


trainer.train(check_point_exists())
trainer.save_model(MODEL_OUTPUT_DIR)

In [14]:
def format_chat_input(input, tokenizer):
    chatml_input = format_chatml(input, None)
    formatted_input = tokenizer.apply_chat_template(chatml_input, tokenize=False, add_generation_prompt=True)
    return formatted_input

In [ ]:
# 使用原有模型进行对话
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
original_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # 混合精度
    device_map="auto",  # 自动分配到 GPU
    cache_dir=MODEL_CACHE_DIR
)
original_model.eval()
pipe = pipeline(
    "text-generation",
    model=original_model,
    tokenizer=tokenizer,
    return_full_text=False,
)

print("开始对话！输入 'exit' 结束对话。")
while True:
    user_input = input("你: ")
    if user_input.lower() == "exit":
        print("对话结束。")
        break
    formatted_input = format_chat_input(user_input, tokenizer)
    response = pipe(formatted_input, truncation=True, max_length=500)
    print(f"模型: {response[0]['generated_text']}")

In [ ]:
# 使用微调后的模型进行对话
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

original_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    cache_dir=MODEL_CACHE_DIR
)

lora_model = PeftModel.from_pretrained(original_model, MODEL_OUTPUT_DIR)

merged_model = lora_model.merge_and_unload()

merged_model.eval()
pipe = pipeline(
    "text-generation",
    model=original_model,
    tokenizer=tokenizer,
    return_full_text=False,
)

print("开始对话！输入 'exit' 结束对话。")
while True:
    user_input = input("你: ")
    if user_input.lower() == "exit":
        print("对话结束。")
        break
    formatted_input = format_chat_input(user_input, tokenizer)
    response = pipe(formatted_input, truncation=True, max_length=500)
    print(f"模型: {response[0]['generated_text']}")